# 🏆 FIFA World Cup Prediction — Notebook 2: Feature Engineering

**What this notebook produces:**

| Feature group | Features built |
|---|---|
| Team strength | Elo rating going into each tournament |
| Pre-tournament form | Win rate & goal diff from last 10 WC matches |
| Squad profile | Avg age, avg WC experience, squad depth by position |
| Tournament history | Past WC appearances, best-ever stage reached |
| Target encoding | `stage_rank` as ordinal target (1=group stage → 7=winner) |

**All data scraped automatically. No uploads needed.**


## 📦 1. Install & Import

In [ ]:
!pip install requests pandas numpy matplotlib seaborn scikit-learn --quiet

import requests, io, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")
print("✅ Ready")

## 🌐 2. Fetch Raw Data

In [ ]:
BASE = "https://raw.githubusercontent.com/jfjelstul/worldcup/master/data-csv/"

def fetch(filename):
    r = requests.get(BASE + filename, timeout=15)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), low_memory=False)
    print(f"  ✅ {filename:<35} {df.shape[0]:>5,} rows × {df.shape[1]} cols")
    return df

print("Fetching data...")
raw_matches      = fetch("matches.csv")
raw_team_app     = fetch("team_appearances.csv")
raw_squads       = fetch("squads.csv")
raw_players      = fetch("players.csv")
raw_tournaments  = fetch("tournaments.csv")
raw_award_winners= fetch("award_winners.csv")
print("\n✅ All files loaded")

## 🗺️ 3. Shared Reference Maps

In [ ]:
# Men's tournaments only
men_tourns = raw_tournaments[
    raw_tournaments["tournament_name"].str.contains("Men", na=False)
].copy()

YEAR_MAP    = men_tourns.set_index("tournament_id")["year"].to_dict()
WINNER_MAP  = men_tourns.set_index("tournament_id")["winner"].to_dict()
WC_YEARS    = sorted(men_tourns["year"].tolist())

# Stage rank: higher = further in tournament
STAGE_RANK = {
    "group stage": 1, "second group stage": 2, "final round": 2,
    "round of 16": 3, "quarter-final": 4, "quarter-finals": 4,
    "semi-final": 5,  "semi-finals": 5,
    "third-place match": 6, "final": 7
}

# Filter all tables to men's WC
def men_only(df):
    df = df.copy()
    df["year"] = df["tournament_id"].map(YEAR_MAP)
    return df[df["year"].notna()].copy()

matches   = men_only(raw_matches)
team_app  = men_only(raw_team_app)
squads    = men_only(raw_squads)

print(f"Men's WC matches  : {len(matches):,}")
print(f"Men's team apps   : {len(team_app):,}")
print(f"Men's squad entries: {len(squads):,}")
print(f"Tournaments covered: {sorted(matches['year'].unique().astype(int))}")

## ⚡ Feature 1 — Elo Ratings

We compute Elo from scratch using all WC match history.

**Why Elo instead of FIFA rankings?**
- FIFA rankings are slow to update and politically influenced
- Elo updates after every match using margin of victory
- Standard K-factor = 32 for international football

**Key property:** We compute the Elo *entering* each tournament, not during it.
This avoids data leakage — we can't use information from the tournament we're predicting.


In [ ]:
def compute_elo(matches_df, k=32, initial=1000):
    """
    Compute Elo ratings from WC match history.
    Returns a dict: {(year, team_name): elo_before_tournament}
    """
    elo = {}          # team -> current elo
    elo_history = {}  # (year, team) -> elo at start of that tournament

    def get_elo(team):
        return elo.get(team, initial)

    def expected(a, b):
        return 1 / (1 + 10 ** ((b - a) / 400))

    def update(team, actual, expected_score, k):
        return get_elo(team) + k * (actual - expected_score)

    # Sort by year then match_date so we process chronologically
    df = matches_df.copy()
    df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")
    df = df.sort_values(["year", "match_date"]).reset_index(drop=True)

    for year, yr_group in df.groupby("year"):
        # Snapshot Elo BEFORE this tournament starts for all teams
        teams_this_year = set(yr_group["home_team_name"]) | set(yr_group["away_team_name"])
        for team in teams_this_year:
            elo_history[(int(year), team)] = get_elo(team)

        # Now process each match and update Elo
        for _, row in yr_group.iterrows():
            home = row["home_team_name"]
            away = row["away_team_name"]
            hg   = row["home_team_score"]
            ag   = row["away_team_score"]

            if pd.isna(hg) or pd.isna(ag):
                continue

            exp_h = expected(get_elo(home), get_elo(away))
            exp_a = 1 - exp_h

            if hg > ag:
                act_h, act_a = 1, 0
            elif hg < ag:
                act_h, act_a = 0, 1
            else:
                act_h, act_a = 0.5, 0.5

            # Margin of victory multiplier (standard football Elo)
            goal_diff = abs(hg - ag)
            mov = np.log(goal_diff + 1) + 1 if goal_diff > 0 else 1

            elo[home] = update(home, act_h, exp_h, k * mov)
            elo[away] = update(away, act_a, exp_a, k * mov)

    return elo_history

elo_history = compute_elo(matches)
print(f"✅ Elo computed for {len(elo_history):,} (year, team) pairs")

# Build elo DataFrame
elo_df = pd.DataFrame([
    {"year": yr, "team_name": team, "elo_pre_tournament": round(elo_val, 1)}
    for (yr, team), elo_val in elo_history.items()
]).sort_values(["year","elo_pre_tournament"], ascending=[True, False]).reset_index(drop=True)

print("\nTop 5 Elo ratings per recent tournament:")
display(elo_df[elo_df["year"] >= 2010].groupby("year").head(5)[
    ["year","team_name","elo_pre_tournament"]
].reset_index(drop=True))

In [ ]:
# Sanity check: do higher Elo teams win tournaments?
winners_elo = elo_df.copy()
winners_elo["tournament_winner"] = winners_elo.apply(
    lambda r: WINNER_MAP.get(f"WC-{int(r['year'])}", None), axis=1
)
winners_elo["is_winner"] = winners_elo["team_name"] == winners_elo["tournament_winner"]

# Rank each team within their tournament by Elo
winners_elo["elo_rank"] = winners_elo.groupby("year")["elo_pre_tournament"].rank(
    ascending=False, method="min"
)

winner_ranks = winners_elo[winners_elo["is_winner"]][["year","team_name","elo_pre_tournament","elo_rank"]]
print("Tournament winners and their Elo rank (1 = highest Elo going in):")
display(winner_ranks.sort_values("year"))

avg_rank = winner_ranks["elo_rank"].mean()
top3_pct = (winner_ranks["elo_rank"] <= 3).mean() * 100
print(f"\nAvg Elo rank of winners: {avg_rank:.1f}")
print(f"% of winners in top-3 Elo: {top3_pct:.0f}%")
print(f"\n→ Interpretation: Elo is {'a strong' if top3_pct > 50 else 'a weak'} predictor alone.")

## 📈 Feature 2 — Pre-Tournament Form

We compute each team's form **within the previous World Cup** as a proxy
for their momentum going into the next one.

**Why not use all international matches?**
The full international results dataset isn't reliably available as a free open file.
Using WC-only history is a valid proxy — it captures peak performance, not friendlies.

**Features computed:**
- `form_win_rate` — win rate in the previous WC (0 if first appearance)
- `form_goal_diff_per_game` — avg goal difference per match in previous WC
- `form_goals_scored_per_game` — attacking output
- `prev_wc_stage_rank` — how far they went last time


In [ ]:
def compute_form_features(team_app_df):
    """Compute per-team per-tournament form from previous WC performance."""

    df = team_app_df.copy()
    df["year"] = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)

    # Aggregate previous WC stats per team
    prev_stats = df.groupby(["year","team_name"]).agg(
        matches_played   = ("match_id", "count"),
        wins             = ("win",       "sum"),
        draws            = ("draw",      "sum"),
        goals_for        = ("goals_for", "sum"),
        goals_against    = ("goals_against", "sum"),
        max_stage_rank   = ("stage_rank", "max"),
    ).reset_index()

    prev_stats["win_rate"]           = prev_stats["wins"] / prev_stats["matches_played"]
    prev_stats["goal_diff_per_game"] = ((prev_stats["goals_for"] - prev_stats["goals_against"])
                                         / prev_stats["matches_played"])
    prev_stats["goals_per_game"]     = prev_stats["goals_for"] / prev_stats["matches_played"]

    # Shift to get PREVIOUS tournament stats
    prev_stats = prev_stats.sort_values(["team_name","year"])
    prev_stats["prev_wc_year"]             = prev_stats.groupby("team_name")["year"].shift(1)
    prev_stats["form_win_rate"]            = prev_stats.groupby("team_name")["win_rate"].shift(1)
    prev_stats["form_goal_diff_per_game"]  = prev_stats.groupby("team_name")["goal_diff_per_game"].shift(1)
    prev_stats["form_goals_per_game"]      = prev_stats.groupby("team_name")["goals_per_game"].shift(1)
    prev_stats["prev_stage_rank"]          = prev_stats.groupby("team_name")["max_stage_rank"].shift(1)

    # Fill NaN (first-time WC teams) with 0
    for col in ["form_win_rate","form_goal_diff_per_game","form_goals_per_game","prev_stage_rank"]:
        prev_stats[col] = prev_stats[col].fillna(0)

    form = prev_stats[[
        "year","team_name",
        "form_win_rate","form_goal_diff_per_game",
        "form_goals_per_game","prev_stage_rank"
    ]].copy()

    print(f"✅ Form features: {len(form):,} rows")
    return form

form_features = compute_form_features(team_app)
display(form_features[form_features["year"] == 2022].sort_values("form_win_rate", ascending=False).head(8))

## 👥 Feature 3 — Squad Profile Features

**Features:**
- `squad_avg_age` — mean age of squad at tournament start
- `squad_avg_experience` — mean number of previous WCs per player (proxy for international caps)
- `squad_size` — total players registered
- `n_forwards`, `n_midfielders`, `n_defenders`, `n_goalkeepers` — positional depth


In [ ]:
def compute_squad_features(squads_df, players_df, tournaments_df):
    sq = squads_df.copy()
    sq["year"] = sq["year"].astype(int)

    # Join birth date from players
    pl = players_df[["player_id","birth_date","count_tournaments","list_tournaments"]].copy()
    pl["birth_date"] = pd.to_datetime(pl["birth_date"], errors="coerce")
    sq = sq.merge(pl, on="player_id", how="left")

    # Tournament start date for age calculation
    t_dates = (
        tournaments_df[tournaments_df["tournament_name"].str.contains("Men", na=False)]
        .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
        [["tournament_id","start_date"]]
    )
    sq = sq.merge(t_dates, on="tournament_id", how="left")
    sq["age"] = ((sq["start_date"] - sq["birth_date"]).dt.days / 365.25)

    # WC experience: how many previous WCs has this player appeared in?
    # Derived from list_tournaments (e.g. "2014, 2018") vs current year
    def count_prev_wc(row):
        try:
            prev = [int(y.strip()) for y in str(row["list_tournaments"]).split(",")]
            return sum(1 for y in prev if y < int(row["year"]))
        except:
            return 0

    sq["prev_wc_count"] = sq.apply(count_prev_wc, axis=1)

    # Position mapping
    pos_map = {
        "goalkeeper": "n_goalkeepers",
        "defender":   "n_defenders",
        "midfielder": "n_midfielders",
        "forward":    "n_forwards",
    }

    agg = sq.groupby(["year","team_name"]).agg(
        squad_size      = ("player_id", "count"),
        squad_avg_age   = ("age",           "mean"),
        squad_min_age   = ("age",           "min"),
        squad_max_age   = ("age",           "max"),
        squad_avg_exp   = ("prev_wc_count", "mean"),
    ).reset_index()

    # Positional counts
    for pos, col in pos_map.items():
        pos_counts = (
            sq[sq["position_name"].str.lower().str.contains(pos, na=False)]
            .groupby(["year","team_name"])
            .size().reset_index(name=col)
        )
        agg = agg.merge(pos_counts, on=["year","team_name"], how="left")

    for col in pos_map.values():
        agg[col] = agg[col].fillna(0).astype(int)

    agg["squad_avg_age"] = agg["squad_avg_age"].round(1)
    agg["squad_avg_exp"] = agg["squad_avg_exp"].round(2)

    print(f"✅ Squad features: {len(agg):,} rows")
    return agg

squad_features = compute_squad_features(squads, raw_players, raw_tournaments)
display(squad_features[squad_features["year"] == 2022].sort_values("squad_avg_exp", ascending=False).head(8))

## 📜 Feature 4 — Tournament History Features

**Features:**
- `wc_appearances` — how many World Cups this team has appeared in (up to this year)
- `best_stage_ever` — furthest stage ever reached (before this tournament)
- `is_host` — whether this team is the host nation (home advantage)
- `won_last_wc` — whether this team won the previous tournament


In [ ]:
def compute_history_features(team_app_df, tournaments_df):
    df = team_app_df.copy()
    df["year"] = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)

    # Max stage per team per tournament
    max_stage = df.groupby(["year","team_name"])["stage_rank"].max().reset_index()
    max_stage = max_stage.sort_values(["team_name","year"])

    # Cumulative appearances and best stage BEFORE this tournament
    records = []
    for team, grp in max_stage.groupby("team_name"):
        grp = grp.sort_values("year").reset_index(drop=True)
        for i, row in grp.iterrows():
            past = grp[grp["year"] < row["year"]]
            records.append({
                "year":            row["year"],
                "team_name":       team,
                "wc_appearances":  len(past),           # appearances BEFORE this WC
                "best_stage_ever": past["stage_rank"].max() if len(past) > 0 else 0,
            })

    hist = pd.DataFrame(records)

    # Host nation flag
    host_map = (
        tournaments_df[tournaments_df["tournament_name"].str.contains("Men", na=False)]
        .set_index("year")["host_country"].to_dict()
    )
    hist["is_host"] = hist.apply(
        lambda r: 1 if host_map.get(int(r["year"])) == r["team_name"] else 0, axis=1
    )

    # Won last WC flag
    winner_by_year = {int(y): w for y, w in
                      zip(men_tourns["year"], men_tourns["winner"])}
    hist["won_last_wc"] = hist.apply(
        lambda r: 1 if winner_by_year.get(int(r["year"]) - 4) == r["team_name"] else 0,
        axis=1
    )

    hist["best_stage_ever"] = hist["best_stage_ever"].fillna(0)
    print(f"✅ History features: {len(hist):,} rows")
    return hist

history_features = compute_history_features(team_app, raw_tournaments)
display(history_features[history_features["year"] == 2022].sort_values("wc_appearances", ascending=False).head(8))

## 🎯 Feature 5 — Target: Stage Reached (Ordinal)

Encode *how far each team got* as an ordinal target variable.

| stage_rank | Stage |
|---|---|
| 1 | Group Stage (eliminated) |
| 2 | Second group stage / Final round |
| 3 | Round of 16 |
| 4 | Quarter-finals |
| 5 | Semi-finals |
| 6 | Third-place match |
| 7 | **Winner** |


In [ ]:
def compute_targets(team_app_df):
    df = team_app_df.copy()
    df["year"]       = df["year"].astype(int)
    df["stage_rank"] = df["stage_name"].str.lower().map(STAGE_RANK)

    targets = df.groupby(["year","team_name"]).agg(
        stage_rank      = ("stage_rank", "max"),
    ).reset_index()

    targets["won_tournament"] = targets.apply(
        lambda r: 1 if WINNER_MAP.get(f"WC-{r['year']}") == r["team_name"] else 0,
        axis=1
    )

    print(f"✅ Targets: {len(targets):,} rows")
    print(f"   stage_rank distribution:")
    print(targets["stage_rank"].value_counts().sort_index().to_string())
    return targets

targets = compute_targets(team_app)

## 🔗 6. Merge All Features → Master Feature Table

In [ ]:
def build_master_features(elo_df, form_features, squad_features,
                           history_features, targets):
    # Start from targets (every team every tournament)
    df = targets.copy()

    df = df.merge(elo_df,           on=["year","team_name"], how="left")
    df = df.merge(form_features,    on=["year","team_name"], how="left")
    df = df.merge(squad_features,   on=["year","team_name"], how="left")
    df = df.merge(history_features, on=["year","team_name"], how="left")

    # Fill remaining nulls
    fill_zero = ["elo_pre_tournament","form_win_rate","form_goal_diff_per_game",
                 "form_goals_per_game","prev_stage_rank","squad_avg_exp",
                 "wc_appearances","best_stage_ever","is_host","won_last_wc"]
    for col in fill_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Fill squad age with median (missing = very old era)
    if "squad_avg_age" in df.columns:
        df["squad_avg_age"] = df["squad_avg_age"].fillna(df["squad_avg_age"].median())

    df = df.sort_values(["year","team_name"]).reset_index(drop=True)
    print(f"✅ Master feature table: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"\nFeature columns:")
    feat_cols = [c for c in df.columns if c not in ["year","team_name","tournament_id"]]
    for c in feat_cols:
        null_pct = df[c].isna().mean()*100
        print(f"  {c:<35} nulls: {null_pct:.1f}%")
    return df

master = build_master_features(elo_df, form_features, squad_features,
                                history_features, targets)
display(master.head(5))

## 📊 7. Feature Analysis — Correlation with Stage Reached

In [ ]:
feature_cols = [
    "elo_pre_tournament","form_win_rate","form_goal_diff_per_game",
    "form_goals_per_game","prev_stage_rank","squad_avg_age","squad_avg_exp",
    "wc_appearances","best_stage_ever","is_host","won_last_wc"
]
feature_cols = [c for c in feature_cols if c in master.columns]

corr = master[feature_cols + ["stage_rank"]].corr()["stage_rank"].drop("stage_rank").sort_values(ascending=False)

print("Correlation with stage_rank (how far team progressed):")
print(corr.round(3).to_string())
print("\n→ Features with |corr| > 0.15 are worth keeping.")
print(f"→ Features with |corr| < 0.05 are likely noise for this dataset size.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Feature Distributions — Winners vs Others", fontsize=14)

plot_features = [
    ("elo_pre_tournament",       "Elo Rating"),
    ("form_win_rate",            "Previous WC Win Rate"),
    ("form_goal_diff_per_game",  "Previous WC Goal Diff/Game"),
    ("squad_avg_age",            "Squad Average Age"),
    ("squad_avg_exp",            "Squad WC Experience"),
    ("wc_appearances",           "WC Appearances"),
]

for ax, (col, title) in zip(axes.flat, plot_features):
    if col not in master.columns:
        ax.set_visible(False)
        continue
    for won, grp in master.groupby("won_tournament"):
        label = "Winner" if won else "Other"
        color = "#FFD700" if won else "#4A90D9"
        ax.hist(grp[col].dropna(), bins=20, alpha=0.7, color=color,
                label=label, edgecolor="white", density=True)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: feature correlations with each other
fig, ax = plt.subplots(figsize=(12, 9))
corr_matrix = master[feature_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix\n(check for multicollinearity before modelling)", fontsize=12)
plt.tight_layout()
plt.show()
print("\n⚠️  Correlated features (|r| > 0.7) may need to be dropped or combined.")
print("   High correlation = redundant information = can hurt some models.")

## 👤 8. Player-Level Features (for Award Prediction)

These feed the Golden Boot, Best Player, and Young Player models.


In [ ]:
raw_goals = pd.read_csv(
    io.StringIO(requests.get(BASE + "goals.csv", timeout=15).text)
)
raw_goals["year"] = raw_goals["tournament_id"].map(YEAR_MAP)
goals = raw_goals[raw_goals["year"].notna()].copy()
goals["year"] = goals["year"].astype(int)

# Goals per player per tournament (excluding own goals)
player_goals = (
    goals[~goals["own_goal"].astype(bool)]
    .groupby(["year","player_id","team_name"]).agg(
        goals_scored   = ("goal_id", "count"),
        penalty_goals  = ("penalty", "sum"),
        open_play_goals= ("penalty", lambda x: (~x.astype(bool)).sum()),
    ).reset_index()
)

# Player appearances (minutes proxy = matches started)
raw_app = pd.read_csv(
    io.StringIO(requests.get(BASE + "player_appearances.csv", timeout=15).text)
)
raw_app["year"] = raw_app["tournament_id"].map(YEAR_MAP)
p_app = raw_app[raw_app["year"].notna()].copy()
p_app["year"] = p_app["year"].astype(int)

player_app = p_app.groupby(["year","player_id","team_name"]).agg(
    matches_played  = ("match_id", "count"),
    matches_started = ("starter",  "sum"),
    position_name   = ("position_name", "first"),
).reset_index()

# Merge with biographical data (age)
bio = raw_players[["player_id","family_name","given_name","birth_date","count_tournaments"]].copy()
bio["player_name"] = bio["given_name"].str.strip() + " " + bio["family_name"].str.strip()
bio["birth_date"]  = pd.to_datetime(bio["birth_date"], errors="coerce")

t_dates = (
    raw_tournaments[raw_tournaments["tournament_name"].str.contains("Men", na=False)]
    .assign(start_date=lambda d: pd.to_datetime(d["start_date"], errors="coerce"))
    .set_index("year")["start_date"].to_dict()
)

# Build player feature table
player_features = player_app.merge(player_goals, on=["year","player_id","team_name"], how="left")
player_features = player_features.merge(bio, on="player_id", how="left")

# Age at tournament
player_features["tournament_start"] = player_features["year"].map(t_dates)
player_features["age_at_tournament"] = (
    (player_features["tournament_start"] - player_features["birth_date"]).dt.days / 365.25
).round(1)

# WC experience
def prev_wcs(row):
    try:
        years = [int(y.strip()) for y in str(row["count_tournaments"]).split(",")]
        return sum(1 for y in years if y < row["year"])
    except:
        return 0

# Use count_tournaments as a number directly (it's a count not a list in this dataset)
player_features["wc_experience"] = player_features["count_tournaments"].fillna(0)

# Fill nulls
player_features["goals_scored"]    = player_features["goals_scored"].fillna(0).astype(int)
player_features["penalty_goals"]   = player_features["penalty_goals"].fillna(0).astype(int)
player_features["open_play_goals"] = player_features["open_play_goals"].fillna(0).astype(int)

# Award flags
raw_aw = pd.read_csv(io.StringIO(requests.get(BASE + "award_winners.csv", timeout=15).text))
raw_aw["year"] = raw_aw["tournament_id"].map(YEAR_MAP)
raw_aw = raw_aw[raw_aw["year"].notna()].copy()

AWARD_IDS = {"A-1":"won_best_player","A-4":"won_golden_boot","A-8":"won_young_player"}
player_features["won_best_player"]  = False
player_features["won_golden_boot"]  = False
player_features["won_young_player"] = False

for _, row in raw_aw.iterrows():
    col = AWARD_IDS.get(row["award_id"])
    if not col: continue
    yr  = row["year"]
    pid = row["player_id"]
    mask = (player_features["year"] == yr) & (player_features["player_id"] == pid)
    player_features.loc[mask, col] = True

print(f"✅ Player features: {player_features.shape[0]:,} rows × {player_features.shape[1]} cols")
print(f"   Golden Boot flags : {player_features['won_golden_boot'].sum()}")
print(f"   Best Player flags : {player_features['won_best_player'].sum()}")
print(f"   Young Player flags: {player_features['won_young_player'].sum()}")
print(f"\nTop scorers 2022:")
display(player_features[player_features["year"]==2022].nlargest(5,"goals_scored")[
    ["player_name","team_name","goals_scored","penalty_goals","matches_played","age_at_tournament"]
])

## 💾 9. Save Feature Tables

In [ ]:
from google.colab import files as colab_files

master.to_csv("team_features.csv", index=False)
player_features.to_csv("player_features.csv", index=False)

colab_files.download("team_features.csv")
colab_files.download("player_features.csv")
print("✅ Downloaded team_features.csv")
print("✅ Downloaded player_features.csv")

## ✅ Done — Feature Engineering Complete

### What you built

| File | Rows | Key features |
|---|---|---|
| `team_features.csv` | ~500 (team × tournament) | Elo, form, squad profile, history, stage_rank target |
| `player_features.csv` | ~11,000 (player × tournament) | Goals, age, experience, position, award flags |

### What to check before modelling

**From the correlation output:**
- Which features correlate strongest with `stage_rank`?
- Are any two features correlated > 0.7 with each other? If yes, drop one.
- Does `elo_pre_tournament` dominate, or do squad features add independent signal?

**From the histograms:**
- Do winner distributions clearly separate from others on any feature?
- Any features where winners and others look identical? Those won't help the model.

### Next notebook — Baseline Model
1. Train/test split by year (not random — leakage risk)
2. Poisson regression for match score prediction
3. Logistic regression for tournament winner probability
4. Monte Carlo simulation (10,000 tournament runs)
5. Evaluate: does the model beat picking the highest-Elo team?
